In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")
tags = pd.read_csv("data/tags.csv")

In [3]:
print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [4]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)

print("\nMovies columns:")
print(movies.columns)

print("\nRatings columns:")
print(ratings.columns)

Movies shape: (9742, 3)
Ratings shape: (100836, 4)

Movies columns:
Index(['movieId', 'title', 'genres'], dtype='object')

Ratings columns:
Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')


In [6]:
print("Number of movies:", movies["movieId"].nunique())
print("Number of users:", ratings["userId"].nunique())
print("Number of ratings:", len(ratings))

Number of movies: 9742
Number of users: 610
Number of ratings: 100836


In [7]:
print(ratings["rating"].describe())

count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64


In [8]:
movie_rating_counts = ratings.groupby("movieId").size().reset_index(name="num_ratings")
top_rated_count = movie_rating_counts.sort_values("num_ratings", ascending=False).head(10)

top_movies = top_rated_count.merge(movies, on="movieId")
print(top_movies[["title", "num_ratings"]])

                                       title  num_ratings
0                        Forrest Gump (1994)          329
1           Shawshank Redemption, The (1994)          317
2                        Pulp Fiction (1994)          307
3           Silence of the Lambs, The (1991)          279
4                         Matrix, The (1999)          278
5  Star Wars: Episode IV - A New Hope (1977)          251
6                       Jurassic Park (1993)          238
7                          Braveheart (1995)          237
8          Terminator 2: Judgment Day (1991)          224
9                    Schindler's List (1993)          220


In [9]:
movies["genres"] = movies["genres"].fillna("")
movies["title"] = movies["title"].fillna("")

In [10]:
print(movies.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64


In [11]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["genres"])

In [12]:
print(tfidf_matrix.shape)

(9742, 23)


In [13]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(cosine_sim.shape)

(9742, 9742)


In [14]:
indices = pd.Series(movies.index, index=movies["title"]).drop_duplicates()

In [15]:
print(indices.head())

title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64


In [16]:
def recommend_movies(title, cosine_sim=cosine_sim):
    if title not in indices:
        return f"Movie '{title}' not found in dataset."
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip the first one because it's the same movie
    sim_scores = sim_scores[1:11]
    
    movie_indices = [i[0] for i in sim_scores]
    
    return movies.iloc[movie_indices][["title", "genres"]]

In [17]:
recommend_movies("Toy Story (1995)")

,title,genres
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
6194,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
7760,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy
8219,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy


In [18]:
print(movies.columns)

Index(['movieId', 'title', 'genres'], dtype='object')


In [19]:
# Remove old num_ratings columns if they already exist
movies = movies.drop(columns=["num_ratings"], errors="ignore")
movies = movies.drop(columns=["num_ratings_x", "num_ratings_y"], errors="ignore")

# Recreate rating counts
rating_counts = ratings.groupby("movieId").size().reset_index(name="num_ratings")

# Merge
movies = movies.merge(rating_counts, on="movieId", how="left")

# Fill missing values
movies["num_ratings"] = movies["num_ratings"].fillna(0)

# Check result
print(movies[["title", "num_ratings"]].head())

                                title  num_ratings
0                    Toy Story (1995)        215.0
1                      Jumanji (1995)        110.0
2             Grumpier Old Men (1995)         52.0
3            Waiting to Exhale (1995)          7.0
4  Father of the Bride Part II (1995)         49.0


In [20]:
def recommend_movies_popular(title, cosine_sim=cosine_sim):
    if title not in indices:
        return f"Movie '{title}' not found in dataset."
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:30]  # take more, then filter
    
    movie_indices = [i[0] for i in sim_scores]
    recs = movies.iloc[movie_indices][["title", "genres", "num_ratings"]].copy()
    
    recs = recs.sort_values("num_ratings", ascending=False).head(10)
    return recs

In [21]:
recommend_movies_popular("Toy Story (1995)")

,title,genres,num_ratings
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,132.0
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,97.0
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,45.0
8900,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy,43.0
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,37.0
8357,The Lego Movie (2014),Action|Adventure|Animation|Children|Comedy|Fan...,31.0
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,21.0
3230,Atlantis: The Lost Empire (2001),Adventure|Animation|Children|Fantasy,19.0
1577,"Lord of the Rings, The (1978)",Adventure|Animation|Children|Fantasy,15.0
6944,Ponyo (Gake no ue no Ponyo) (2008),Adventure|Animation|Children|Fantasy,11.0


In [22]:
def print_recommendations(title):
    result = recommend_movies_popular(title)
    
    if isinstance(result, str):
        print(result)
    else:
        print(f"If you liked '{title}', you might also like:\n")
        for i, row in enumerate(result.itertuples(), start=1):
            print(f"{i}. {row.title} | Genres: {row.genres} | Ratings count: {int(row.num_ratings)}")

In [23]:
print_recommendations("Toy Story (1995)")

If you liked 'Toy Story (1995)', you might also like:

1. Monsters, Inc. (2001) | Genres: Adventure|Animation|Children|Comedy|Fantasy | Ratings count: 132
2. Toy Story 2 (1999) | Genres: Adventure|Animation|Children|Comedy|Fantasy | Ratings count: 97
3. Antz (1998) | Genres: Adventure|Animation|Children|Comedy|Fantasy | Ratings count: 45
4. Inside Out (2015) | Genres: Adventure|Animation|Children|Comedy|Drama|Fantasy | Ratings count: 43
5. Emperor's New Groove, The (2000) | Genres: Adventure|Animation|Children|Comedy|Fantasy | Ratings count: 37
6. The Lego Movie (2014) | Genres: Action|Adventure|Animation|Children|Comedy|Fantasy | Ratings count: 31
7. Shrek the Third (2007) | Genres: Adventure|Animation|Children|Comedy|Fantasy | Ratings count: 21
8. Atlantis: The Lost Empire (2001) | Genres: Adventure|Animation|Children|Fantasy | Ratings count: 19
9. Lord of the Rings, The (1978) | Genres: Adventure|Animation|Children|Fantasy | Ratings count: 15
10. Ponyo (Gake no ue no Ponyo) (2008) |

In [24]:
def find_movie(partial_title):
    matches = movies[movies["title"].str.contains(partial_title, case=False, na=False)]
    return matches[["title"]].head(20)

In [28]:
find_movie("Titanic")
find_movie("Toy Story")

,title
0,Toy Story (1995)
2355,Toy Story 2 (1999)
7355,Toy Story 3 (2010)


In [31]:
tag_text = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x.astype(str))).reset_index()
tag_text.rename(columns={"tag": "all_tags"}, inplace=True)

movies = movies.merge(tag_text, on="movieId", how="left")
movies["all_tags"] = movies["all_tags"].fillna("")

In [26]:
# improve function

In [32]:
def clean_title(title):
    return re.sub(r"\(\d{4}\)", "", title).strip().lower()

movies["clean_title"] = movies["title"].apply(clean_title)

# -----------------------------
# 7. Create combined text features
# -----------------------------
movies["combined_features"] = (
    movies["clean_title"] + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["all_tags"]
)

# -----------------------------
# 8. Weighted rating
# -----------------------------
C = movies["avg_rating"].mean()
m = movies["num_ratings"].quantile(0.75)

movies["weighted_rating"] = (
    (movies["num_ratings"] / (movies["num_ratings"] + m)) * movies["avg_rating"] +
    (m / (movies["num_ratings"] + m)) * C
)


KeyError: 'avg_rating'

In [ ]:
def recommend_movies_advanced(title, top_n=10, min_ratings=20):
    if title not in indices:
        return f"Movie '{title}' not found. Use find_movie() first."

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # skip itself
    sim_scores = sim_scores[1:50]

    movie_indices = [i[0] for i in sim_scores]
    similarity_values = [i[1] for i in sim_scores]

    recs = movies.iloc[movie_indices][[
        "title", "genres", "year", "avg_rating", "num_ratings"
    ]].copy()

    recs["similarity"] = similarity_values

    # filter weak / unpopular movies
    recs = recs[recs["num_ratings"] >= min_ratings]

    # weighted ranking
    recs["score"] = (
        recs["similarity"] * 0.6 +
        (recs["avg_rating"] / 5.0) * 0.25 +
        np.log1p(recs["num_ratings"]) / np.log1p(recs["num_ratings"].max()) * 0.15
    )

    recs = recs.sort_values("score", ascending=False).head(top_n)

    return recs[["title", "year", "genres", "avg_rating", "num_ratings", "similarity", "score"]].reset_index(drop=True)

In [ ]:
rating_stats = ratings.groupby("movieId").agg(
    num_ratings=("rating", "count"),
    avg_rating=("rating", "mean"),
    rating_std=("rating", "std")
).reset_index()

movies = movies.merge(rating_stats, on="movieId", how="left")

In [ ]:
C = movies["avg_rating"].mean()
m = movies["num_ratings"].quantile(0.75)

movies["weighted_rating"] = (
    (movies["num_ratings"] / (movies["num_ratings"] + m)) * movies["avg_rating"] +
    (m / (movies["num_ratings"] + m)) * C
)

In [ ]:
movies["combined_features"] = (
    movies["clean_title"] + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["all_tags"]
)

In [ ]:
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")

latest_rating = ratings.groupby("movieId")["timestamp"].max().reset_index()
latest_rating.rename(columns={"timestamp": "last_rating_date"}, inplace=True)

movies = movies.merge(latest_rating, on="movieId", how="left")

In [ ]:
movies["num_genres"] = movies["genres"].apply(lambda x: len(x.split("|")))

In [ ]:
movies = movies[movies["num_ratings"] >= 20]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

movies[["num_ratings", "avg_rating"]] = scaler.fit_transform(
    movies[["num_ratings", "avg_rating"]]
)

In [ ]:
movies["popularity_score"] = (
    0.5 * movies["weighted_rating"] +
    0.5 * np.log1p(movies["num_ratings"])
)

In [ ]:
recs["final_score"] = (
    recs["similarity"] * 0.6 +
    recs["weighted_rating"] * 0.3 +
    np.log1p(recs["num_ratings"]) * 0.1
)

In [ ]:
recs = recs[recs["num_ratings"] > 20]

In [ ]:
movies.to_csv("movies_enriched.csv", index=False)